In [ ]:
import pandas as pd
import numpy as np
import ollama
import re
import json

from src.io import read_tabular
from pathlib import Path
from tqdm.notebook import tqdm

from pydantic import BaseModel, Field
from typing import List, Optional, Literal

In [ ]:
fp = Path("../../data")
df_uk = read_tabular(fp / "uk_sentences.csv")
input = df_uk['contexted']
devset = input.sample(5)

### Setting up LLM pipeline

In [ ]:
# Model for now: GPT-OSS 20b for local development

GPTSMALL = 'gpt-oss:20b'
ollama.pull(GPTSMALL)

client = ollama.Client()

In [ ]:
def classify_text(text, system_message, model):

  # clean the text 
  text = re.sub(r'\s+', ' ', text).strip()

  # construct input

  messages = [
    # system prompt
    {"role": "system", "content": system_message},
    # user input
    {"role": "user", "content": text},
  ]

  # set some options controlling generation behavior
  opts = {
      'seed': 42,         # seed controlling random number generation and thus stochastic generation
      'temperature': 0.0, # hyper parameter controlling "craetivity", see https://learnprompting.org/docs/basics/configuration_hyperparameters
      #'max_tokens': 3     # maximum numbers of tokens to generate in completion
  }
  response = client.chat(
    model=model,
    messages=messages,
    options=opts,
    format=json_schema
  )
  
  # NOTE: this changed slightly compared to using `openai` Client
  result = response.message.content.strip()
  
  return result

For structured output, we want to define a JSON scheme to return each element of the core sentence individually. The JSON contains the following elements:  

- Sentence: the sentence the model has coded (without its context) 
- Type: the type of core sentence, pre-defined to be one of "actor-actor" or "actor-issue". 
- Subject: the subject of the core sentence
- Direction: the direction of the core sentence, pre-defined to be one of "support", "opposition", "ambivalent".
- Object: the object of the sentence, being either an actor or an issue
- Reference: an issue that is being referenced in an actor-actor sentence. Empty in case of actor-issue sentences.

In [30]:
class CoreSent(BaseModel):
    type: Literal['actor-actor', 'actor_issue', 'NA'] = Field(..., description = "The category of core sentence detected")
    subject: str = Field(..., description="The subject of the core sentence")
    direction: Literal["support", "opposition", "ambivalent", 'NA'] = Field(..., description = "The stance taken by the actor towards the subject")
    object: Optional[str] = Field(None, description = "The object of the core sentence")
    issue: Optional[str] = Field(None, description = "An issue being mentioned in an actor-actor sentence")

class CSResponse(BaseModel):
    sentence: str = Field(..., description="The grammatical sentence you coded")
    core_sents: Optional[List[CoreSent]] = Field(
        None,
        description="List of core sentences extracted from the sentence. Leave empty if none are detected."
    )

class fullResponse(BaseModel):
    contexted: str = Field(..., description = "The input given to the model")
    response: List[CSResponse] = Field(
        ...,
        description = "Output of the model"
    )

json_schema = CSResponse.model_json_schema()

In [ ]:
def transform_and_save(raw_outputs: List[dict], output_file: str = "llm_outputs.json") -> None:
    """
    Transforms raw Ollama outputs into validated CSResponse objects and saves them as a JSON file.

    Args:
        raw_outputs: List of raw responses from Ollama (e.g., your `out` list).
        output_file: Path to the output JSON file.
    """
    validated_outputs = []

    for raw in raw_outputs:
        try:
            # Extract the content from the Ollama response
            content = raw.get("message", {}).get("content", "{}")
            parsed = json.loads(content)
            # Validate and parse using Pydantic
            validated = CSResponse(**parsed)
            validated_outputs.append(validated.model_dump())
        except (ValidationError, json.JSONDecodeError, AttributeError) as e:
            print(f"Skipping invalid response: {raw}. Error: {e}")
            continue

    # Save to JSON file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(validated_outputs, f, indent=2, ensure_ascii=False)

    print(f"Successfully saved {len(validated_outputs)} validated outputs to {output_file}.")

## Prompts

#### For technical development: simplified baseline prompt

In [ ]:
sysprompt_json = f'''
You are an expert coder with training and expertise in analysing political claims. You follow British politics to the level of an interested, engaged daily news reader, and know the most important politicians, parties, and issues in the United Kingdom in late 2025/early 2026. Your task is to analyze sentences from newspaper articles and extract the following variables according to the instructions provided:

1. Sentence: The sentence you are coding
2. Type: Identify whether the sentence describes an "actor-actor" or an "actor-issue" relationship.
3. Subject: Extract the name of the organization affiliated with the person making a claim or taking an action in the sentence.
4. Direction: Code the stance of the Subject towards the Object - one of "support", "opposition", or "ambiguous".
5. Object: Extract the name of the organization affiliated with the object of the sentence.
6. Issue reference: If the sentence references a political issue (either as the target of an actor-issue sentence or as a reference in an actor-actor sentence), describe it here.

Extract all variables present in the sentence. If there is no relationship expressed in a sentence, do not code the variables. If there is more than one relationship described in one sentence, extract the variables for each relationship separately.

##Detailed instructions

- Type: The type depends on the object of the sentence. If the subject is referring to another political or societal actor, such as a political party, a government, a protest movement, or a business interest, code "actor-actor". If the subject is referring to a political issue or position, code "actor-issue".
- Subject: The subject of the sentence should always be a political or societal actor, such as a party, politician, protest movement, business interest, etc. We are interested in the semantic subject expressing a stance, even if this differs from the grammatical subject. When there are multiple possible organisations for one actor, prioritise political parties when possible.
- Direction: Determine whether the subject supports or opposes the object or issue, and code accordingly. Only use "ambiguous" if the stance is truly unclear.
- Object: The object of a sentence should always be a political or societal actor. We are interested here in the semantic object, i.e. the target of the subject's stance, even if this differs from the grammatical object. When there are multiple possible organisations for one object, prioritise political parties when possible.
- Issue reference: If you have identified an actor-issue-relation, this is the issue the subject takes a stance towards. If you have identified an actor-actor-relation, this is any issue referenced to explain or justify the subject's stance towards the object.

##Input

You are given up to five sentences published in a British newspaper, one of which is marked with > <. Code only the marked sentence, but use the other sentences to provide context to the marked sentence.

##Output

Return the extracted variables according to the following JSON scheme:

{json.dumps(json_schema)}
'''

## Classifying sentences

In [ ]:
for text in devset:
    messages = [
        {"role": "system", "content": sysprompt_json},
        {"role": "user", "content": text}
    ]

    opts = {
        "seed": 42,
        "temperature": 0.0
    }

    response = client.chat(
        model = GPTSMALL,
        messages = messages,
        options = opts,
        format = json_schema
    )
    
output = CSResponse.model_validate_json(resp)

In [ ]:
out = []

for text in devset:
    messages = [
        {"role": "system", "content": sysprompt_json},
        {"role": "user", "content": text}
    ]

    opts = {
        "seed": 42,
        "temperature": 0.0
    }

    response = client.chat(
        model = GPTSMALL,
        messages = messages,
        options = opts,
        format = json_schema
    )
    
    out.append({
        "contexted": text,
        "response": response
    })

In [33]:
print(out)

transform_and_save(out, "output_baseline.json")

[{'contexted': 'Absolutely. Do we have to go to more scale and more speed? Absolutely. > But you cannot argue that the Paris Agreement is unsuccessful." < Graphic   Ed Miliband hailed the "action and atmosphere" at the climate summit', 'response': ChatResponse(model='gpt-oss:20b', created_at='2026-04-09T14:58:13.9038774Z', done=True, done_reason='stop', total_duration=129909320200, load_duration=7534001600, prompt_eval_count=1604, prompt_eval_duration=166993100, eval_count=21, eval_duration=3362201200, message=Message(role='assistant', content='{"sentence":"But you cannot argue that the Paris Agreement is unsuccessful.","core_sents":null}', thinking='We need to code only the marked sentence: "But you cannot argue that the Paris Agreement is unsuccessful." The other sentences provide context but not coded.\n\nWe need to extract variables: Sentence, core_sents array. Determine if there\'s a relationship. The sentence: "But you cannot argue that the Paris Agreement is unsuccessful." This 

NameError: name 'ValidationError' is not defined